<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Amortization-of-Risk-Indicators/blob/main/Amortization_Exp5_3_C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================================
# EXPERIMENT 5.3-C
# COMPLETE RUNTIME + ACCURACY-DEPENDENT BREAK-EVEN
#
# RESUMABLE / GOOGLE-DRIVE / CPU-ONLY / PUBLICATION VERSION
#
# R_GRID = (500, 1000, 2000, 5000, 10000)
# validation = 200
# test       = 300
#
# COMPLETE OUTPUTS:
#   p(0),...,p(N),p(>N)
#   rho(c) = P(C>c)
#   E(tau)
#   Var(tau)
#
# PERSISTENCE:
#   1. exact teacher data -> permanent Drive chunks
#   2. neural training    -> checkpoint every 5 epochs
#   3. completed R fits   -> permanent Drive models/results
#   4. runtime benchmark  -> checkpoint per configuration
#   5. final tables/figures -> permanent Drive results
#
# AFTER COLAB DISCONNECT:
#   -> reconnect
#   -> rerun THIS SAME CELL
#   -> completed work is skipped automatically
#
# IMPORTANT FIXES:
#   - training_sec is stored in final model files and preserved on reload
#   - break-even always includes C_teach + C_training
#   - runtime bootstrap resamples independent configurations, not technical repeats
#
# CPU ONLY
# NO MATRIX INVERSE
# =====================================================================================


# =====================================================================================
# 0. GOOGLE DRIVE
# =====================================================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)


# =====================================================================================
# 1. CPU ENVIRONMENT
# =====================================================================================

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"


# =====================================================================================
# 2. IMPORTS
# =====================================================================================

import time
import math
import random
import pickle
import shutil

from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd

from scipy import sparse
from scipy.sparse.linalg import splu
from scipy.stats import qmc

from joblib import Parallel, delayed

import torch
import torch.nn as nn

import matplotlib.pyplot as plt


# =====================================================================================
# 3. CONFIG
# =====================================================================================

@dataclass
class C:

    seed:int = 20260820

    beta:Tuple[float,float] = (.30,1.50)
    gamma:Tuple[float,float] = (.20,1.00)
    omega:Tuple[float,float] = (.02,.50)
    frac:Tuple[float,float] = (.02,.20)

    trainN:Tuple[int,...] = tuple(
        range(40,401,20)
    )

    width:int = 128
    depth:int = 3

    batch:int = 64
    epochs:int = 500

    lr:float = 1e-3
    wd:float = 1e-6

    patience:int = 20
    delta:float = 1e-6
    clip:float = 5.

    lambda_tau:float = .02
    refine:int = 3


cfg = C()


# =====================================================================================
# 4. EXPERIMENT SETTINGS
# =====================================================================================

R_GRID = (
    500,
    1000,
    2000,
    5000,
    10000
)

RMAX = max(
    R_GRID
)

N_VAL = 200
N_TEST = 300


# Accuracy requirements
EPS = (
    .05,
    .025,
    .01
)


# Runtime grid
BENCH_N = (
    50,
    75,
    100,
    125,
    150,
    175,
    200,
    225,
    250,
    275,
    300,
    325,
    350,
    375,
    400
)


# -------------------------------------------------------------------------
# Independent benchmark configurations per N
# -------------------------------------------------------------------------

BENCH_PER_N = 5


# Technical repetitions within each independent configuration
EXACT_REPEATS = 5

NEURAL_BLOCKS = 5
NEURAL_REPEATS = 100


# Paired configuration-level bootstrap
BOOTSTRAP_B = 5000


# Persistence granularity
EXACT_CHUNK = 50
CHECKPOINT_EVERY = 5


Nscale = max(
    cfg.trainN
)


TEST_N = tuple(
    range(40,401,10)
)


# =====================================================================================
# 5. PERMANENT GOOGLE DRIVE STORAGE
# =====================================================================================

ROOT = Path(
    "/content/drive/MyDrive/"
    "StatisticalLearning/"
    "Experiment_5_3C_CPU_publication_resumable_v5"
)


CACHE = ROOT / "cache"

EXACT_CACHE = CACHE / "exact"
MODEL_CACHE = CACHE / "models"
CHECKPOINT_CACHE = CACHE / "training_checkpoints"
RUNTIME_CACHE = CACHE / "runtime"

OUT = ROOT / "results"


for d in (
    ROOT,
    CACHE,
    EXACT_CACHE,
    MODEL_CACHE,
    CHECKPOINT_CACHE,
    RUNTIME_CACHE,
    OUT
):

    d.mkdir(
        parents=True,
        exist_ok=True
    )


print("="*105)
print("PERMANENT GOOGLE DRIVE DIRECTORY")
print(ROOT)
print("="*105)


# =====================================================================================
# 6. ATOMIC SAVES
#
# Write to temporary file first and rename only after a successful write.
# This protects against a disconnect occurring during a save.
# =====================================================================================

def atomic_pickle(
    obj,
    path
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )


    with open(
        tmp,
        "wb"
    ) as f:

        pickle.dump(
            obj,
            f,
            pickle.HIGHEST_PROTOCOL
        )


    os.replace(
        tmp,
        path
    )


def atomic_torch_save(
    obj,
    path
):

    path = Path(path)


    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )


    torch.save(
        obj,
        tmp
    )


    os.replace(
        tmp,
        path
    )


def safe_pickle_load(
    path,
    default=None
):

    try:

        with open(
            path,
            "rb"
        ) as f:

            return pickle.load(
                f
            )

    except Exception:

        return default


# =====================================================================================
# 7. CPU CONFIGURATION
# =====================================================================================

CPU = (
    os.cpu_count()
    or
    1
)


# Large sparse LU decompositions are memory-intensive.
N_EXACT = max(
    1,
    min(
        2,
        CPU
    )
)


# Small MLP: using all logical cores can be counterproductive.
TORCH_THREADS = max(
    1,
    min(
        8,
        CPU
    )
)


torch.set_num_threads(
    TORCH_THREADS
)


try:

    torch.set_num_interop_threads(
        1
    )

except RuntimeError:

    pass


device = torch.device(
    "cpu"
)


def seed_all(s):

    random.seed(
        s
    )

    np.random.seed(
        s
    )

    torch.manual_seed(
        s
    )


seed_all(
    cfg.seed
)


print(
    "Logical CPU cores:",
    CPU
)

print(
    "Exact-LU workers:",
    N_EXACT
)

print(
    "PyTorch threads:",
    torch.get_num_threads()
)

print(
    "R grid:",
    R_GRID
)

print(
    "Exact training configurations:",
    RMAX
)

print(
    "Validation configurations:",
    N_VAL
)

print(
    "Test configurations:",
    N_TEST
)

print(
    "Benchmark configurations/N:",
    BENCH_PER_N
)

print(
    "Exact timings/configuration:",
    EXACT_REPEATS
)

print(
    "Neural timing blocks/configuration:",
    NEURAL_BLOCKS
)

print(
    "Neural queries/block:",
    NEURAL_REPEATS
)

print(
    "Training checkpoint every:",
    CHECKPOINT_EVERY,
    "epochs"
)

print(
    "GPU used: NO"
)


# =====================================================================================
# 8. RECORD
# =====================================================================================

@dataclass
class Rec:

    b:float
    g:float
    w:float

    N:int
    i0:int

    p:np.ndarray

    mt:float
    vt:float
    tv:bool

    # Complete exact-query time.
    # These values are summed to form C_teach.
    teach_sec:float = 0.


# =====================================================================================
# 9. OPTIONAL MIGRATION FROM OLD /content SESSION
#
# Exact datasets can safely be reused because they contain the same exact targets
# and per-query teach_sec values.
#
# Old neural models are migrated ONLY if a valid positive training time can be
# recovered from the old accuracy table. This avoids the old training_sec=0 bug.
# =====================================================================================

OLD_CACHE = Path(
    "/content/cache_5_3C_CPU_publication"
)

OLD_OUT = Path(
    "/content/results_5_3C_CPU_publication"
)


if OLD_CACHE.exists():

    print(
        "\nOld /content 5.3-C cache detected."
    )


    old_exact_map = {

        "train_R10000_CPU.pkl":
            EXACT_CACHE/"TRAIN_full.pkl",

        "validation_200_CPU.pkl":
            EXACT_CACHE/"VALIDATION_full.pkl",

        "test_300_CPU.pkl":
            EXACT_CACHE/"TEST_full.pkl"
    }


    for oldname,newfile in old_exact_map.items():

        oldfile = (
            OLD_CACHE
            /
            oldname
        )


        if (
            oldfile.exists()
            and
            not newfile.exists()
        ):

            try:

                shutil.copy2(
                    oldfile,
                    newfile
                )

                print(
                    "Migrated exact cache:",
                    oldname
                )

            except Exception as e:

                print(
                    "Could not migrate",
                    oldname,
                    ":",
                    e
                )


    # -------------------------------------------------------------------------
    # Migrate old models only when their measured training times are available.
    # -------------------------------------------------------------------------

    old_acc_file = (
        OLD_OUT
        /
        "accuracy_cost_by_R_CPU.csv"
    )


    old_acc = None


    if old_acc_file.exists():

        try:

            old_acc = pd.read_csv(
                old_acc_file
            )

        except Exception:

            old_acc = None


    if old_acc is not None:

        for R in R_GRID:

            old_model = (
                OLD_CACHE
                /
                f"model_R{R}_CPU.pt"
            )


            new_model = (
                MODEL_CACHE
                /
                f"model_R{R}_FINAL.pt"
            )


            if (
                new_model.exists()
                or
                not old_model.exists()
            ):

                continue


            z = old_acc[
                old_acc["R"]==R
            ]


            if len(z)==0:

                continue


            training_sec = float(
                z.iloc[0][
                    "training_sec"
                ]
            )


            # Never propagate the historical cache bug.
            if (
                not np.isfinite(training_sec)
                or
                training_sec<=0
            ):

                continue


            try:

                ck = torch.load(
                    old_model,
                    map_location="cpu",
                    weights_only=False
                )


                atomic_torch_save(

                    {

                        "h":
                            ck["h"],

                        "t":
                            ck["t"],

                        "best_epoch":
                            ck.get(
                                "best_epoch",
                                -1
                            ),

                        "validation":
                            ck.get(
                                "validation",
                                np.nan
                            ),

                        "training_sec":
                            training_sec

                    },

                    new_model

                )


                print(
                    f"Migrated R={R} model "
                    f"with training_sec={training_sec:.1f}s"
                )


            except Exception as e:

                print(
                    f"Could not migrate R={R} model:",
                    e
                )


# =====================================================================================
# 10. SIRS TOPOLOGY
# =====================================================================================

@lru_cache(None)
def topo(N):

    states = [

        (s,i)

        for i in range(
            1,
            N+1
        )

        for s in range(
            N-i+1
        )

    ]


    ix = {

        x:j

        for j,x
        in enumerate(
            states
        )

    }


    M = len(
        states
    )


    ir=[]; ic=[]; ib=[]
    rr=[]; rc=[]; rb=[]
    wr=[]; wc=[]; wb=[]


    db = np.zeros(
        M
    )

    dg = np.zeros(
        M
    )

    dw = np.zeros(
        M
    )

    qb = np.zeros(
        M
    )


    for j,(s,i) in enumerate(
        states
    ):

        r = (
            N-s-i
        )


        # Infection
        if s:

            ir.append(
                j
            )

            ic.append(
                ix[
                    (s-1,i+1)
                ]
            )

            rate = (
                s*i/N
            )

            ib.append(
                rate
            )

            db[j] = (
                rate
            )


        # Recovery
        dg[j] = (
            i
        )


        if i==1:

            qb[j] = (
                i
            )

        else:

            rr.append(
                j
            )

            rc.append(
                ix[
                    (s,i-1)
                ]
            )

            rb.append(
                i
            )


        # Immunity loss
        if r:

            wr.append(
                j
            )

            wc.append(
                ix[
                    (s+1,i)
                ]
            )

            wb.append(
                r
            )

            dw[j] = (
                r
            )


    A = (
        lambda x,d=float:
        np.asarray(
            x,
            dtype=d
        )
    )


    return (

        ix,
        M,

        A(ir,int),
        A(ic,int),
        A(ib),

        A(rr,int),
        A(rc,int),
        A(rb),

        A(wr,int),
        A(wc,int),
        A(wb),

        db,
        dg,
        dw,
        qb
    )


# =====================================================================================
# 11. ITERATIVE REFINEMENT
# =====================================================================================

def solve_refined(
    A,
    lu,
    b
):

    b = np.asarray(
        b,
        dtype=np.float64
    )


    x = lu.solve(
        b
    )


    for _ in range(
        cfg.refine
    ):

        residual = (
            b
            -
            A@x
        )


        if not np.all(
            np.isfinite(
                residual
            )
        ):

            break


        rel = (

            np.linalg.norm(
                residual,
                np.inf
            )

            /

            max(
                np.linalg.norm(
                    b,
                    np.inf
                ),
                1.
            )

        )


        if rel < 1e-11:

            break


        x += lu.solve(
            residual
        )


    return np.asarray(
        x,
        dtype=np.float64
    )


# =====================================================================================
# 12. COMPLETE EXACT QUERY
# =====================================================================================

def exact_full(
    b,
    g,
    w,
    N,
    i0
):

    (
        ix,
        M,

        ir,
        ic,
        ib,

        rr,
        rc,
        rb,

        wr,
        wc,
        wb,

        db,
        dg,
        dw,
        qb

    ) = topo(
        N
    )


    rows = np.r_[

        ir,
        rr,
        wr,
        np.arange(M)

    ]


    cols = np.r_[

        ic,
        rc,
        wc,
        np.arange(M)

    ]


    vals = np.r_[

        b*ib,

        g*rb,

        w*wb,

        -(
            b*db
            +
            g*dg
            +
            w*dw
        )

    ]


    T = sparse.coo_matrix(

        (
            vals,
            (
                rows,
                cols
            )
        ),

        shape=(
            M,
            M
        ),

        dtype=np.float64

    ).tocsc()


    D1 = sparse.coo_matrix(

        (
            b*ib,
            (
                ir,
                ic
            )
        ),

        shape=(
            M,
            M
        ),

        dtype=np.float64

    ).tocsc()


    D0 = (
        T-D1
    ).tocsc()


    q = (
        g*qb
    )


    initial = ix[
        (
            N-i0,
            i0
        )
    ]


    # =========================================================================
    # A. Infection-count distribution
    # =========================================================================

    A0 = (
        -D0
    ).tocsc()


    lu0 = splu(
        A0,
        permc_spec="COLAMD"
    )


    bb = lu0.solve(
        q
    )


    v = np.zeros(
        M,
        dtype=np.float64
    )


    v[
        initial
    ] = 1.


    p = np.zeros(
        N+2,
        dtype=np.float64
    )


    D1T = (
        D1.T.tocsr()
    )


    for k in range(
        N+1
    ):

        p[k] = (
            v@bb
        )


        y = lu0.solve(
            v,
            trans="T"
        )


        v = np.asarray(
            D1T@y
        ).ravel()


    # overflow
    p[-1] = (
        v.sum()
    )


    p[
        np.abs(p)<1e-12
    ] = 0.


    p = np.maximum(
        p,
        0.
    )


    mass = (
        p.sum()
    )


    if (
        not np.isfinite(
            mass
        )
        or
        mass<=0
    ):

        raise RuntimeError(
            f"Invalid exact PMF: "
            f"N={N}, i0={i0}"
        )


    p /= (
        mass
    )


    rho = np.flip(

        np.cumsum(

            np.flip(
                p[1:]
            )

        )

    )


    # =========================================================================
    # B. Extinction-time moments
    # =========================================================================

    A = (
        -T
    ).tocsc()


    d = np.abs(
        A.diagonal()
    )


    scaling = (

        1.

        /

        np.maximum(
            d,
            np.finfo(float).tiny
        )

    )


    As = (

        sparse.diags(
            scaling
        )

        @

        A

    ).tocsc()


    errors = []


    for ordering in (
        "COLAMD",
        "MMD_AT_PLUS_A"
    ):

        try:

            lu = splu(
                As,
                permc_spec=ordering
            )


            def moment_solve(
                rhs
            ):

                return solve_refined(

                    As,

                    lu,

                    scaling
                    *
                    np.asarray(
                        rhs,
                        dtype=np.float64
                    )

                )


            m1 = moment_solve(

                np.ones(
                    M,
                    dtype=np.float64
                )

            )


            mean = float(
                m1[
                    initial
                ]
            )


            if (
                not np.all(
                    np.isfinite(
                        m1
                    )
                )
                or
                mean<=0
            ):

                raise ArithmeticError(
                    f"Invalid E(tau)={mean}"
                )


            m2 = moment_solve(
                2.*m1
            )


            second = float(
                m2[
                    initial
                ]
            )


            if (
                not np.all(
                    np.isfinite(
                        m2
                    )
                )
                or
                second<=0
            ):

                raise ArithmeticError(
                    f"Invalid E(tau^2)={second}"
                )


            raw = (

                np.longdouble(
                    second
                )

                -

                np.longdouble(
                    mean
                )**2

            )


            tol = (
                1e-8
                *
                max(
                    abs(second),
                    mean*mean,
                    1.
                )
            )


            if (
                np.isfinite(raw)
                and
                raw>=-tol
            ):

                var = max(
                    float(raw),
                    0.
                )


            else:

                C = (
                    T.tocoo()
                )


                off = (
                    C.row
                    !=
                    C.col
                )


                source = np.bincount(

                    C.row[
                        off
                    ],

                    weights=(

                        C.data[
                            off
                        ]

                        *

                        (
                            m1[
                                C.col[
                                    off
                                ]
                            ]

                            -

                            m1[
                                C.row[
                                    off
                                ]
                            ]
                        )**2

                    ),

                    minlength=M

                ).astype(
                    np.float64
                )


                source += (
                    q
                    *
                    m1
                    *
                    m1
                )


                if (
                    not np.all(
                        np.isfinite(
                            source
                        )
                    )
                    or
                    source.min() < -1e-8
                ):

                    raise ArithmeticError(
                        "Invalid variance-system RHS."
                    )


                vv = moment_solve(

                    np.maximum(
                        source,
                        0.
                    )

                )


                var = float(
                    vv[
                        initial
                    ]
                )


            if (
                not np.isfinite(
                    var
                )
                or
                var<0
            ):

                raise ArithmeticError(
                    f"Invalid Var(tau)={var}"
                )


            return (
                p,
                rho,
                mean,
                var,
                True
            )


        except Exception as e:

            errors.append(
                f"{ordering}: {e}"
            )


    # Distribution remains valid even if moment calculation is unresolved.
    return (
        p,
        rho,
        np.nan,
        np.nan,
        False
    )


# =====================================================================================
# 13. EXPERIMENTAL DESIGN
# =====================================================================================

def design(
    n,
    Ns,
    seed
):

    U = qmc.LatinHypercube(

        d=4,

        seed=seed

    ).random(
        n
    )


    scale = lambda x,a: (

        a[0]
        +
        (a[1]-a[0])*x

    )


    beta = scale(
        U[:,0],
        cfg.beta
    )


    gamma = scale(
        U[:,1],
        cfg.gamma
    )


    omega = scale(
        U[:,2],
        cfg.omega
    )


    frac = scale(
        U[:,3],
        cfg.frac
    )


    Nv = np.tile(

        np.asarray(
            Ns
        ),

        math.ceil(
            n/len(Ns)
        )

    )[:n]


    rng = np.random.default_rng(
        seed+99
    )


    rng.shuffle(
        Nv
    )


    i0 = np.asarray([

        int(
            np.clip(
                round(
                    frac[j]
                    *
                    Nv[j]
                ),
                2,
                Nv[j]
            )
        )

        for j in range(n)

    ])


    for N in Ns:

        idx = np.where(
            Nv==N
        )[0]


        if len(idx)==0:

            continue


        k = max(
            1,
            round(
                .25*len(idx)
            )
        )


        i0[
            rng.choice(
                idx,
                k,
                replace=False
            )
        ] = 1


    return [

        (
            float(beta[j]),
            float(gamma[j]),
            float(omega[j]),
            int(Nv[j]),
            int(i0[j])
        )

        for j in range(n)

    ]


# =====================================================================================
# 14. EXACT TARGET WORKER
# =====================================================================================

def _one_exact(
    j,
    x
):

    t0 = (
        time.perf_counter()
    )


    p,rho,m,v,ok = exact_full(
        *x
    )


    elapsed = (
        time.perf_counter()
        -
        t0
    )


    return (

        j,

        Rec(
            *x,
            p,
            m,
            v,
            ok,
            elapsed
        )

    )


# =====================================================================================
# 15. RESUMABLE EXACT DATASET
#
# Each chunk is permanently saved to Google Drive.
#
# If Colab disconnects, only the currently unfinished chunk is lost.
# =====================================================================================

def make_cached_resumable(
    configs,
    name
):

    full_file = (
        EXACT_CACHE
        /
        f"{name}_full.pkl"
    )


    # -------------------------------------------------------------------------
    # Entire dataset already complete
    # -------------------------------------------------------------------------

    if full_file.exists():

        ans = safe_pickle_load(
            full_file
        )


        if (
            ans is not None
            and
            len(ans)==len(configs)
        ):

            print(
                f"{name}: full Drive cache loaded "
                f"({len(ans):,})"
            )


            return ans


    folder = (
        EXACT_CACHE
        /
        name
    )


    folder.mkdir(
        parents=True,
        exist_ok=True
    )


    ans = []


    for start in range(
        0,
        len(configs),
        EXACT_CHUNK
    ):

        end = min(
            start+EXACT_CHUNK,
            len(configs)
        )


        file = (

            folder

            /

            f"chunk_{start:05d}_{end:05d}.pkl"

        )


        part = None


        # ---------------------------------------------------------------------
        # Load completed chunk
        # ---------------------------------------------------------------------

        if file.exists():

            part = safe_pickle_load(
                file
            )


            if (
                part is not None
                and
                len(part)==end-start
            ):

                print(
                    f"{name} "
                    f"{start:5d}:{end:5d} | "
                    "Drive cache"
                )

            else:

                part = None


        # ---------------------------------------------------------------------
        # Compute missing chunk
        # ---------------------------------------------------------------------

        if part is None:

            jobs = list(
                enumerate(
                    configs[
                        start:end
                    ]
                )
            )


            # Within this chunk, process expensive N first.
            jobs.sort(
                key=lambda z:
                z[1][3],
                reverse=True
            )


            print(
                f"{name} "
                f"{start:5d}:{end:5d} | "
                f"{N_EXACT} exact workers"
            )


            t0 = (
                time.perf_counter()
            )


            result = Parallel(

                n_jobs=N_EXACT,

                backend="threading"

            )(

                delayed(
                    _one_exact
                )(
                    j,
                    x
                )

                for j,x in jobs

            )


            result.sort(
                key=lambda z:
                z[0]
            )


            part = [

                r

                for _,r
                in result

            ]


            atomic_pickle(
                part,
                file
            )


            print(
                f"  saved permanently | "
                f"{time.perf_counter()-t0:.1f}s"
            )


        ans.extend(
            part
        )


    # -------------------------------------------------------------------------
    # Complete compact cache
    # -------------------------------------------------------------------------

    atomic_pickle(
        ans,
        full_file
    )


    print(
        f"{name}: COMPLETE and permanently cached."
    )


    return ans


# =====================================================================================
# 16. NETWORKS
# =====================================================================================

def mlp(
    din,
    dout
):

    L = []

    d = (
        din
    )


    for _ in range(
        cfg.depth
    ):

        L += [

            nn.Linear(
                d,
                cfg.width
            ),

            nn.SiLU()

        ]


        d = (
            cfg.width
        )


    L.append(

        nn.Linear(
            d,
            dout
        )

    )


    return nn.Sequential(
        *L
    )


class HazardNet(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()

        self.net = mlp(
            6,
            1
        )


    def forward(
        self,
        x
    ):

        return torch.sigmoid(

            self.net(
                x
            ).squeeze(-1)

        )


class TauNet(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()

        self.net = mlp(
            5,
            2
        )


    def forward(
        self,
        x
    ):

        return torch.nn.functional.softplus(

            self.net(
                x
            )

        )


# =====================================================================================
# 17. PACK DATA BY N
# =====================================================================================

def pack(
    records
):

    groups = {}


    for j,r in enumerate(
        records
    ):

        groups.setdefault(
            r.N,
            []
        ).append(
            j
        )


    P = {}


    for N,idx in groups.items():

        idx = np.asarray(
            idx,
            dtype=int
        )


        rr = [

            records[j]

            for j in idx

        ]


        B = (
            len(rr)
        )


        K = (
            N+1
        )


        beta = torch.tensor(

            [
                r.b
                for r in rr
            ],

            dtype=torch.float32

        )[:,None]


        gamma = torch.tensor(

            [
                r.g
                for r in rr
            ],

            dtype=torch.float32

        )[:,None]


        omega = torch.tensor(

            [
                r.w
                for r in rr
            ],

            dtype=torch.float32

        )[:,None]


        ns = torch.full(

            (
                B,
                1
            ),

            N/Nscale,

            dtype=torch.float32

        )


        i0 = torch.tensor(

            [
                r.i0/N
                for r in rr
            ],

            dtype=torch.float32

        )[:,None]


        c = (

            torch.arange(
                K,
                dtype=torch.float32
            )

            /

            N

        )[None,:]


        Xh = torch.stack(

            [

                beta.expand(B,K),
                gamma.expand(B,K),
                omega.expand(B,K),
                ns.expand(B,K),
                i0.expand(B,K),
                c.expand(B,K)

            ],

            dim=2

        ).contiguous()


        Yp = torch.from_numpy(

            np.stack(
                [
                    r.p
                    for r in rr
                ]
            ).astype(
                np.float32,
                copy=False
            )

        )


        Xt = torch.column_stack(

            [

                beta[:,0],
                gamma[:,0],
                omega[:,0],
                ns[:,0],
                i0[:,0]

            ]

        ).contiguous()


        mask = torch.tensor(

            [
                r.tv
                for r in rr
            ],

            dtype=torch.bool

        )


        Yt_np = np.zeros(
            (
                B,
                2
            ),
            dtype=np.float32
        )


        for j,r in enumerate(
            rr
        ):

            if r.tv:

                Yt_np[j] = np.log1p(

                    [
                        r.mt,
                        r.vt
                    ]

                )


        Yt = torch.from_numpy(
            Yt_np
        )


        P[N] = {

            "Xh":
                Xh,

            "Yp":
                Yp,

            "Xt":
                Xt,

            "Yt":
                Yt,

            "mask":
                mask,

            "n":
                B

        }


    return P


# =====================================================================================
# 18. HAZARD -> PMF / TAIL
# =====================================================================================

def reconstruct(
    h
):

    B = (
        h.shape[0]
    )


    before = torch.cat(

        [

            torch.ones(
                (B,1),
                dtype=h.dtype
            ),

            torch.cumprod(
                1-h[:,:-1],
                dim=1
            )

        ],

        dim=1

    )


    return torch.cat(

        [

            before*h,

            torch.prod(
                1-h,
                dim=1,
                keepdim=True
            )

        ],

        dim=1

    )


def tail(
    p
):

    return torch.flip(

        torch.cumsum(

            torch.flip(
                p[:,1:],
                dims=[1]
            ),

            dim=1

        ),

        dims=[1]

    )


# =====================================================================================
# 19. LOSS
# =====================================================================================

def batch_loss(
    hnet,
    tnet,
    G,
    idx
):

    X = (
        G["Xh"][idx]
    )


    Y = (
        G["Yp"][idx]
    )


    B,K,_ = (
        X.shape
    )


    H = hnet(

        X.reshape(
            B*K,
            6
        )

    ).reshape(
        B,
        K
    )


    P = reconstruct(
        H
    )


    Lp = torch.sum(

        (
            P-Y
        )**2,

        dim=1

    ).mean()


    Lrho = torch.mean(

        (

            tail(P)

            -

            tail(Y)

        )**2,

        dim=1

    ).mean()


    valid = (
        G["mask"][idx]
    )


    if torch.any(
        valid
    ):

        target = (
            G["Yt"][idx][valid]
        )


        pred = tnet(

            G["Xt"][idx][valid]

        )


        Ltau = torch.sum(

            (
                pred-target
            )**2

            /

            (
                1
                +
                target*target
            ),

            dim=1

        ).mean()


    else:

        Ltau = torch.tensor(
            0.,
            dtype=torch.float32
        )


    return (

        Lp

        +

        Lrho

        +

        cfg.lambda_tau
        *
        Ltau

    )


# =====================================================================================
# 20. BATCH SCHEDULE
# =====================================================================================

def batches(
    P,
    rng,
    shuffle=True
):

    ans = []


    for N,G in P.items():

        idx = np.arange(
            G["n"]
        )


        if shuffle:

            rng.shuffle(
                idx
            )


        for s in range(
            0,
            len(idx),
            cfg.batch
        ):

            ans.append(

                (

                    N,

                    idx[
                        s:
                        s+cfg.batch
                    ]

                )

            )


    if shuffle:

        rng.shuffle(
            ans
        )


    return ans


# =====================================================================================
# 21. VALIDATION
# =====================================================================================

@torch.no_grad()
def validation(
    hnet,
    tnet,
    V
):

    hnet.eval()
    tnet.eval()


    rng = np.random.default_rng(
        1
    )


    total = 0.
    n = 0


    for N,idx in batches(
        V,
        rng,
        shuffle=False
    ):

        L = batch_loss(

            hnet,
            tnet,

            V[N],

            idx

        )


        total += (
            L.item()
            *
            len(idx)
        )


        n += (
            len(idx)
        )


    return (
        total/n
    )


# =====================================================================================
# 22. RESUMABLE NEURAL TRAINING
#
# FIX:
# training_sec is stored in FINAL model.
#
# Thus loading a completed model never silently changes training cost to zero.
# =====================================================================================

def fit_resumable(
    TR,
    VA,
    R
):

    final_file = (

        MODEL_CACHE

        /

        f"model_R{R}_FINAL.pt"

    )


    checkpoint_file = (

        CHECKPOINT_CACHE

        /

        f"model_R{R}_CHECKPOINT.pt"

    )


    hnet = HazardNet()
    tnet = TauNet()


    # -------------------------------------------------------------------------
    # Finished model
    # -------------------------------------------------------------------------

    if final_file.exists():

        ck = torch.load(

            final_file,

            map_location="cpu",

            weights_only=False

        )


        hnet.load_state_dict(
            ck["h"]
        )


        tnet.load_state_dict(
            ck["t"]
        )


        training_sec = float(
            ck.get(
                "training_sec",
                np.nan
            )
        )


        if (
            not np.isfinite(
                training_sec
            )
            or
            training_sec<=0
        ):

            raise RuntimeError(

                f"R={R}: final model exists but "
                f"training_sec is missing/invalid.\n"
                f"Delete only:\n{final_file}\n"
                f"and rerun so the training cost is measured correctly."

            )


        print(
            f"R={R}: FINAL model loaded | "
            f"best epoch={ck['best_epoch']} | "
            f"training={training_sec:.1f}s"
        )


        return (
            hnet,
            tnet,
            training_sec
        )


    # -------------------------------------------------------------------------
    # New training setup
    # -------------------------------------------------------------------------

    seed_all(
        cfg.seed+7001
    )


    pars = (

        list(
            hnet.parameters()
        )

        +

        list(
            tnet.parameters()
        )

    )


    opt = torch.optim.AdamW(

        pars,

        lr=cfg.lr,

        weight_decay=cfg.wd

    )


    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(

        opt,

        factor=.5,

        patience=15

    )


    rng = np.random.default_rng(
        cfg.seed+77
    )


    start_epoch = 1

    best = np.inf

    best_h = None
    best_t = None

    best_epoch = 0
    wait = 0

    previous_elapsed = 0.


    # -------------------------------------------------------------------------
    # Resume interrupted training
    # -------------------------------------------------------------------------

    if checkpoint_file.exists():

        try:

            ck = torch.load(

                checkpoint_file,

                map_location="cpu",

                weights_only=False

            )


            hnet.load_state_dict(
                ck["h_current"]
            )


            tnet.load_state_dict(
                ck["t_current"]
            )


            opt.load_state_dict(
                ck["optimizer"]
            )


            sch.load_state_dict(
                ck["scheduler"]
            )


            start_epoch = (
                ck["epoch"]
                +
                1
            )


            best = (
                ck["best"]
            )


            best_epoch = (
                ck["best_epoch"]
            )


            best_h = (
                ck["best_h"]
            )


            best_t = (
                ck["best_t"]
            )


            wait = (
                ck["wait"]
            )


            previous_elapsed = float(
                ck.get(
                    "elapsed_sec",
                    0.
                )
            )


            rng.bit_generator.state = (
                ck["rng_state"]
            )


            if "torch_rng_state" in ck:

                torch.set_rng_state(
                    ck["torch_rng_state"]
                )


            print(
                f"R={R}: RESUMING from epoch "
                f"{start_epoch}"
            )


            print(
                f"  best epoch={best_epoch} | "
                f"best val={best:.4e} | "
                f"wait={wait}"
            )


        except Exception as e:

            print(
                f"R={R}: checkpoint load failed."
            )

            print(
                e
            )

            print(
                "Starting this R from epoch 1."
            )


    session_start = (
        time.perf_counter()
    )


    # -------------------------------------------------------------------------
    # Training
    # -------------------------------------------------------------------------

    for epoch in range(
        start_epoch,
        cfg.epochs+1
    ):

        hnet.train()
        tnet.train()


        for N,idx in batches(
            TR,
            rng,
            shuffle=True
        ):

            opt.zero_grad(
                set_to_none=True
            )


            L = batch_loss(

                hnet,
                tnet,

                TR[N],

                idx

            )


            if not torch.isfinite(
                L
            ):

                raise RuntimeError(
                    f"R={R}: non-finite loss "
                    f"at epoch={epoch}"
                )


            L.backward()


            torch.nn.utils.clip_grad_norm_(

                pars,

                cfg.clip

            )


            opt.step()


        v = validation(
            hnet,
            tnet,
            VA
        )


        sch.step(
            v
        )


        # ---------------------------------------------------------------------
        # Best model
        # ---------------------------------------------------------------------

        if (
            best_h is None
            or
            v < best-cfg.delta
        ):

            best = (
                v
            )

            best_epoch = (
                epoch
            )

            wait = (
                0
            )


            best_h = {

                k:
                x.detach().clone()

                for k,x
                in hnet.state_dict().items()

            }


            best_t = {

                k:
                x.detach().clone()

                for k,x
                in tnet.state_dict().items()

            }


        else:

            wait += (
                1
            )


        # ---------------------------------------------------------------------
        # Permanent checkpoint
        # ---------------------------------------------------------------------

        if (
            epoch==1
            or
            epoch%CHECKPOINT_EVERY==0
        ):

            elapsed = (

                previous_elapsed

                +

                (
                    time.perf_counter()
                    -
                    session_start
                )

            )


            atomic_torch_save(

                {

                    "epoch":
                        epoch,

                    "h_current":
                        hnet.state_dict(),

                    "t_current":
                        tnet.state_dict(),

                    "optimizer":
                        opt.state_dict(),

                    "scheduler":
                        sch.state_dict(),

                    "best":
                        best,

                    "best_epoch":
                        best_epoch,

                    "best_h":
                        best_h,

                    "best_t":
                        best_t,

                    "wait":
                        wait,

                    "rng_state":
                        rng.bit_generator.state,

                    "torch_rng_state":
                        torch.get_rng_state(),

                    "elapsed_sec":
                        elapsed

                },

                checkpoint_file

            )


            print(
                f"R={R} | "
                f"epoch={epoch:3d} | "
                f"val={v:.4e} | "
                f"best={best:.4e} | "
                f"best_epoch={best_epoch} | "
                f"wait={wait} | "
                "CHECKPOINT"
            )


        elif epoch%20==0:

            print(
                f"R={R} | "
                f"epoch={epoch:3d} | "
                f"val={v:.4e} | "
                f"best={best:.4e} | "
                f"wait={wait}"
            )


        if wait>=cfg.patience:

            print(
                f"R={R}: early stopping "
                f"at epoch {epoch}"
            )

            break


    # -------------------------------------------------------------------------
    # Finalize
    # -------------------------------------------------------------------------

    if (
        best_h is None
        or
        best_t is None
    ):

        raise RuntimeError(
            f"R={R}: no valid best model."
        )


    elapsed = (

        previous_elapsed

        +

        (
            time.perf_counter()
            -
            session_start
        )

    )


    hnet.load_state_dict(
        best_h
    )


    tnet.load_state_dict(
        best_t
    )


    atomic_torch_save(

        {

            "h":
                best_h,

            "t":
                best_t,

            "best_epoch":
                best_epoch,

            "validation":
                best,

            "training_sec":
                elapsed

        },

        final_file

    )


    if checkpoint_file.exists():

        checkpoint_file.unlink()


    print(
        f"R={R}: COMPLETE | "
        f"best epoch={best_epoch} | "
        f"training={elapsed:.1f}s"
    )


    return (
        hnet,
        tnet,
        elapsed
    )


# =====================================================================================
# 23. TEST METRICS
# =====================================================================================

@torch.no_grad()
def test_metrics(
    hnet,
    tnet,
    P
):

    hnet.eval()
    tnet.eval()


    E2 = []
    Erho = []


    for N,G in P.items():

        for s in range(
            0,
            G["n"],
            cfg.batch
        ):

            X = G["Xh"][
                s:
                s+cfg.batch
            ]


            Y = G["Yp"][
                s:
                s+cfg.batch
            ]


            B,K,_ = (
                X.shape
            )


            H = hnet(

                X.reshape(
                    B*K,
                    6
                )

            ).reshape(
                B,
                K
            )


            Ph = reconstruct(
                H
            )


            E2.append(

                torch.linalg.vector_norm(
                    Ph-Y,
                    dim=1
                ).numpy()

            )


            Erho.append(

                torch.max(

                    torch.abs(

                        tail(Ph)

                        -

                        tail(Y)

                    ),

                    dim=1

                ).values.numpy()

            )


    return (

        float(
            np.median(
                np.concatenate(
                    E2
                )
            )
        ),

        float(
            np.median(
                np.concatenate(
                    Erho
                )
            )
        )

    )


# =====================================================================================
# 24. DATASETS
# =====================================================================================

print(
    "\nGenerating deterministic designs..."
)


train_design = design(
    RMAX,
    cfg.trainN,
    cfg.seed+1
)


val_design = design(
    N_VAL,
    cfg.trainN,
    cfg.seed+2
)


test_design = design(
    N_TEST,
    TEST_N,
    cfg.seed+3
)


print(
    "\nGenerating/loading exact datasets..."
)


train = make_cached_resumable(
    train_design,
    "TRAIN_R10000"
)


val = make_cached_resumable(
    val_design,
    "VALIDATION_200"
)


test = make_cached_resumable(
    test_design,
    "TEST_300"
)


VA = pack(
    val
)


TE = pack(
    test
)


# =====================================================================================
# 25. PERMANENT ACCURACY PROGRESS
# =====================================================================================

ACC_PROGRESS_FILE = (
    ROOT
    /
    "accuracy_progress.pkl"
)


acc_progress = safe_pickle_load(

    ACC_PROGRESS_FILE,

    default={}

)


if acc_progress is None:

    acc_progress = {}


# =====================================================================================
# 26. LEARNING CURVE + OFFLINE COST
#
# Cteach_sec is the cumulative measured complete exact-query cost
#
#     C_teach(R) = sum_{r=1}^R c_E(N_r).
#
# It is NOT reset when models are loaded from cache.
# =====================================================================================

for R in R_GRID:

    key = (
        int(R)
    )


    model_file = (

        MODEL_CACHE

        /

        f"model_R{R}_FINAL.pt"

    )


    # -------------------------------------------------------------------------
    # Fully completed R
    # -------------------------------------------------------------------------

    if (
        key in acc_progress
        and
        model_file.exists()
    ):

        print(
            f"\nR={R}: accuracy result "
            "already complete — skipping."
        )

        continue


    print(
        "\n"
        +
        "="*90
    )


    print(
        f"TRAINING / EVALUATING R={R}"
    )


    print(
        "="*90
    )


    TR = pack(
        train[:R]
    )


    hnet,tnet,training_sec = fit_resumable(

        TR,
        VA,
        R

    )


    e2,erho = test_metrics(

        hnet,
        tnet,
        TE

    )


    Cteach = float(
        sum(
            x.teach_sec
            for x in train[:R]
        )
    )


    offline = (
        Cteach
        +
        training_sec
    )


    acc_progress[key] = {

        "R":
            R,

        "median_E2":
            e2,

        "median_Erho":
            erho,

        "Cteach_sec":
            Cteach,

        "training_sec":
            training_sec,

        "offline_sec":
            offline

    }


    atomic_pickle(
        acc_progress,
        ACC_PROGRESS_FILE
    )


    print(
        f"R={R} | "
        f"E2={e2:.6g} | "
        f"Erho={erho:.6g} | "
        f"Cteach={Cteach:.1f}s | "
        f"training={training_sec:.1f}s | "
        f"offline={offline:.1f}s"
    )


    del TR
    del hnet
    del tnet


# =====================================================================================
# 27. ACCURACY TABLE
# =====================================================================================

acc = pd.DataFrame(

    [
        acc_progress[R]
        for R in R_GRID
        if R in acc_progress
    ]

)


acc = acc.sort_values(
    "R"
).reset_index(
    drop=True
)


acc.to_csv(

    OUT
    /
    "accuracy_cost_by_R_CPU.csv",

    index=False

)


(
    OUT
    /
    "accuracy_cost_by_R_CPU.tex"
).write_text(

    acc.to_latex(
        index=False,
        float_format="%.5g"
    )

)


print(
    "\nACCURACY / OFFLINE COST\n"
)

print(
    acc.to_string(
        index=False
    )
)


# =====================================================================================
# 28. LOAD RMAX MODEL FOR RUNTIME BENCHMARK
# =====================================================================================

R_RUNTIME = max(
    R_GRID
)


runtime_model_file = (

    MODEL_CACHE

    /

    f"model_R{R_RUNTIME}_FINAL.pt"

)


if not runtime_model_file.exists():

    raise RuntimeError(

        f"R={R_RUNTIME} model is required "
        "before runtime benchmarking."

    )


ck = torch.load(

    runtime_model_file,

    map_location="cpu",

    weights_only=False

)


hmax = HazardNet()
tmax = TauNet()


hmax.load_state_dict(
    ck["h"]
)


tmax.load_state_dict(
    ck["t"]
)


hmax.eval()
tmax.eval()


# =====================================================================================
# 29. COMPLETE SINGLE NEURAL QUERY
# =====================================================================================

@torch.inference_mode()
def neural_query(
    hnet,
    tnet,
    x
):

    b,g,w,N,i0 = (
        x
    )


    K = (
        N+1
    )


    c = (

        torch.arange(
            K,
            dtype=torch.float32
        )

        /

        N

    )


    X = torch.column_stack(

        [

            torch.full(
                (K,),
                b
            ),

            torch.full(
                (K,),
                g
            ),

            torch.full(
                (K,),
                w
            ),

            torch.full(
                (K,),
                N/Nscale
            ),

            torch.full(
                (K,),
                i0/N
            ),

            c

        ]

    ).float()


    H = hnet(
        X
    ).reshape(
        1,
        K
    )


    P = reconstruct(
        H
    )[0]


    RHO = tail(
        P.unsqueeze(0)
    )[0]


    Xt = torch.tensor(

        [

            b,
            g,
            w,
            N/Nscale,
            i0/N

        ],

        dtype=torch.float32

    ).unsqueeze(0)


    z = tnet(
        Xt
    )[0]


    moments = torch.expm1(

        torch.clamp(
            z,
            0,
            80
        )

    )


    return (

        P,
        RHO,
        moments[0],
        moments[1]

    )


# =====================================================================================
# 30. PAIRED CONFIGURATION-LEVEL BOOTSTRAP
#
# Five epidemic configurations are the independent statistical units.
#
# Repeated timings within a configuration are technical replicates and are first
# reduced to a configuration-specific median.
#
# Bootstrap then resamples the five configurations jointly for exact and neural.
# =====================================================================================

def paired_speedup_ci(
    exact_config_times,
    neural_config_times,
    B=5000,
    seed=12345
):

    exact_config_times = np.asarray(
        exact_config_times,
        dtype=np.float64
    )


    neural_config_times = np.asarray(
        neural_config_times,
        dtype=np.float64
    )


    if (
        len(exact_config_times)
        !=
        len(neural_config_times)
    ):

        raise ValueError(
            "Exact/neural configuration counts differ."
        )


    n = (
        len(exact_config_times)
    )


    rng = np.random.default_rng(
        seed
    )


    ratios = np.empty(
        B,
        dtype=np.float64
    )


    for b in range(
        B
    ):

        idx = rng.integers(
            0,
            n,
            size=n
        )


        ratios[b] = (

            np.median(
                exact_config_times[
                    idx
                ]
            )

            /

            np.median(
                neural_config_times[
                    idx
                ]
            )

        )


    return (

        float(
            np.quantile(
                ratios,
                .025
            )
        ),

        float(
            np.quantile(
                ratios,
                .975
            )
        )

    )


# =====================================================================================
# 31. BENCHMARK CONFIGURATIONS
# =====================================================================================

bench_design = design(

    BENCH_PER_N
    *
    len(BENCH_N),

    BENCH_N,

    cfg.seed+50

)


# =====================================================================================
# 32. RESUMABLE PUBLICATION-QUALITY RUNTIME BENCHMARK
#
# Each configuration is saved immediately.
#
# Disconnect during N=400 / configuration 4?
# -> configurations 1-3 remain on Drive.
# =====================================================================================

runtime_rows = []


print(
    "\n"
    +
    "="*105
)


print(
    "PUBLICATION-QUALITY COMPLETE-QUERY CPU RUNTIME BENCHMARK"
)


print(
    "="*105
)


for N in BENCH_N:

    xs = [

        x

        for x in bench_design

        if x[3]==N

    ][:BENCH_PER_N]


    if len(xs)!=BENCH_PER_N:

        raise RuntimeError(
            f"Expected {BENCH_PER_N} "
            f"benchmark configurations at N={N}; "
            f"found {len(xs)}."
        )


    # Exclude topology construction from query timing.
    topo(
        N
    )


    config_results = []


    for config_id,x in enumerate(
        xs
    ):

        file = (

            RUNTIME_CACHE

            /

            f"N{N:03d}_"
            f"config{config_id:02d}.pkl"

        )


        z = None


        if file.exists():

            z = safe_pickle_load(
                file
            )


            if (
                z is not None
                and
                len(
                    z.get(
                        "exact_times",
                        []
                    )
                )==EXACT_REPEATS
                and
                len(
                    z.get(
                        "neural_times",
                        []
                    )
                )==NEURAL_BLOCKS
            ):

                print(
                    f"N={N:3d} | "
                    f"config={config_id+1}/{BENCH_PER_N} | "
                    "Drive cache"
                )

            else:

                z = None


        # ---------------------------------------------------------------------
        # Missing benchmark configuration
        # ---------------------------------------------------------------------

        if z is None:

            print(
                f"N={N:3d} | "
                f"config={config_id+1}/{BENCH_PER_N} | "
                "benchmarking..."
            )


            # -------------------------------------------------------------
            # Warm-up
            #
            # Benchmark therefore represents steady-state repeated queries.
            # -------------------------------------------------------------

            exact_full(
                *x
            )


            for _ in range(
                10
            ):

                neural_query(
                    hmax,
                    tmax,
                    x
                )


            # -------------------------------------------------------------
            # Exact technical repetitions
            # -------------------------------------------------------------

            exact_times = []


            for _ in range(
                EXACT_REPEATS
            ):

                t0 = (
                    time.perf_counter()
                )


                exact_full(
                    *x
                )


                exact_times.append(

                    time.perf_counter()
                    -
                    t0

                )


            # -------------------------------------------------------------
            # Neural technical repetitions
            #
            # Each observation is a 100-query block average.
            # -------------------------------------------------------------

            neural_times = []


            for _ in range(
                NEURAL_BLOCKS
            ):

                t0 = (
                    time.perf_counter()
                )


                for _ in range(
                    NEURAL_REPEATS
                ):

                    neural_query(
                        hmax,
                        tmax,
                        x
                    )


                neural_times.append(

                    (
                        time.perf_counter()
                        -
                        t0
                    )

                    /

                    NEURAL_REPEATS

                )


            z = {

                "N":
                    N,

                "config_id":
                    config_id,

                "x":
                    x,

                "exact_times":
                    np.asarray(
                        exact_times,
                        dtype=np.float64
                    ),

                "neural_times":
                    np.asarray(
                        neural_times,
                        dtype=np.float64
                    )

            }


            atomic_pickle(
                z,
                file
            )


            print(
                f"  exact median="
                f"{np.median(exact_times):.6g}s | "
                f"neural median="
                f"{np.median(neural_times):.6g}s | "
                "saved"
            )


        config_results.append(
            z
        )


    # =========================================================================
    # Configuration-level timing summaries
    # =========================================================================

    exact_config = np.asarray(

        [

            np.median(
                z["exact_times"]
            )

            for z in config_results

        ],

        dtype=np.float64

    )


    neural_config = np.asarray(

        [

            np.median(
                z["neural_times"]
            )

            for z in config_results

        ],

        dtype=np.float64

    )


    # The central runtime and IQR are across independent configurations.
    exact_med = float(
        np.median(
            exact_config
        )
    )


    exact_q1 = float(
        np.quantile(
            exact_config,
            .25
        )
    )


    exact_q3 = float(
        np.quantile(
            exact_config,
            .75
        )
    )


    neural_med = float(
        np.median(
            neural_config
        )
    )


    neural_q1 = float(
        np.quantile(
            neural_config,
            .25
        )
    )


    neural_q3 = float(
        np.quantile(
            neural_config,
            .75
        )
    )


    speedup = (
        exact_med
        /
        neural_med
    )


    lo,hi = paired_speedup_ci(

        exact_config,

        neural_config,

        B=BOOTSTRAP_B,

        seed=cfg.seed+N

    )


    runtime_rows.append(

        [

            N,

            exact_med,
            exact_q1,
            exact_q3,
            exact_q3-exact_q1,

            neural_med,
            neural_q1,
            neural_q3,
            neural_q3-neural_q1,

            speedup,
            lo,
            hi,

            BENCH_PER_N,
            BENCH_PER_N*EXACT_REPEATS,
            BENCH_PER_N*NEURAL_BLOCKS

        ]

    )


    print(
        f"N={N:3d} | "
        f"exact={exact_med:.6g}s "
        f"[{exact_q1:.6g},{exact_q3:.6g}] | "
        f"neural={neural_med:.6g}s "
        f"[{neural_q1:.6g},{neural_q3:.6g}] | "
        f"speedup={speedup:.2f}x "
        f"[95% CI {lo:.2f},{hi:.2f}]"
    )


    # -------------------------------------------------------------------------
    # Save partial runtime table after every completed N
    # -------------------------------------------------------------------------

    partial_runtime = pd.DataFrame(

        runtime_rows,

        columns=[

            "N",

            "exact_sec",
            "exact_q1",
            "exact_q3",
            "exact_IQR",

            "neural_sec",
            "neural_q1",
            "neural_q3",
            "neural_IQR",

            "speedup",
            "speedup_CI_low",
            "speedup_CI_high",

            "n_independent_configs",
            "n_exact_timings",
            "n_neural_timing_blocks"

        ]

    )


    partial_runtime.to_csv(

        OUT
        /
        "runtime_CPU_partial.csv",

        index=False

    )


# =====================================================================================
# 33. FINAL RUNTIME TABLE
# =====================================================================================

runtime = pd.DataFrame(

    runtime_rows,

    columns=[

        "N",

        "exact_sec",
        "exact_q1",
        "exact_q3",
        "exact_IQR",

        "neural_sec",
        "neural_q1",
        "neural_q3",
        "neural_IQR",

        "speedup",
        "speedup_CI_low",
        "speedup_CI_high",

        "n_independent_configs",
        "n_exact_timings",
        "n_neural_timing_blocks"

    ]

)


runtime.to_csv(

    OUT
    /
    "runtime_CPU.csv",

    index=False

)


(
    OUT
    /
    "runtime_CPU.tex"
).write_text(

    runtime.to_latex(
        index=False,
        float_format="%.5g"
    )

)


print(
    "\nCPU RUNTIME TABLE\n"
)


print(
    runtime.to_string(
        index=False
    )
)


# =====================================================================================
# 34. ACCURACY-DEPENDENT BREAK-EVEN
#
# R_epsilon = smallest R attaining median E_rho <= epsilon.
#
# Offline cost:
#
#     C_offline(R)
#       = C_teach(R) + C_training(R).
#
# Break-even:
#
#     B*_epsilon(N)
#       = C_offline(R_epsilon)
#         /
#         [c_E(N)-c_F(N)].
# =====================================================================================

break_even_rows = []


for eps in EPS:

    acceptable = acc[
        acc["median_Erho"]<=eps
    ]


    if len(
        acceptable
    )==0:

        print(
            f"No R satisfies epsilon={eps:g}"
        )

        continue


    selected = acceptable.sort_values(
        "R"
    ).iloc[0]


    R_eps = int(
        selected[
            "R"
        ]
    )


    offline = float(
        selected[
            "offline_sec"
        ]
    )


    # Safety check against the historical cache bug
    expected = (

        float(
            selected[
                "Cteach_sec"
            ]
        )

        +

        float(
            selected[
                "training_sec"
            ]
        )

    )


    if not np.isclose(
        offline,
        expected,
        rtol=1e-10,
        atol=1e-8
    ):

        raise RuntimeError(
            "Offline-cost consistency check failed."
        )


    for _,z in runtime.iterrows():

        exact_sec = float(
            z["exact_sec"]
        )


        neural_sec = float(
            z["neural_sec"]
        )


        if exact_sec<=neural_sec:

            continue


        Bstar = (

            offline

            /

            (
                exact_sec
                -
                neural_sec
            )

        )


        break_even_rows.append(

            [

                eps,

                R_eps,

                int(
                    z["N"]
                ),

                offline,

                float(
                    selected[
                        "Cteach_sec"
                    ]
                ),

                float(
                    selected[
                        "training_sec"
                    ]
                ),

                exact_sec,

                neural_sec,

                Bstar

            ]

        )


br = pd.DataFrame(

    break_even_rows,

    columns=[

        "epsilon",
        "R_epsilon",
        "N",

        "offline_sec",
        "Cteach_sec",
        "training_sec",

        "exact_sec",
        "neural_sec",

        "B_star"

    ]

)


br.to_csv(

    OUT
    /
    "break_even_CPU.csv",

    index=False

)


(
    OUT
    /
    "break_even_CPU.tex"
).write_text(

    br.to_latex(
        index=False,
        float_format="%.5g"
    )

)


print(
    "\nACCURACY-DEPENDENT BREAK-EVEN\n"
)


if len(
    br
):

    print(
        br.to_string(
            index=False
        )
    )


else:

    print(
        "No break-even values available."
    )


# =====================================================================================
# 35. FIGURE 1 — PUBLICATION RUNTIME
#
# Only the essential runtime figure:
#
# A. Complete-query runtime + IQR across independent configurations
# B. Speedup + paired configuration-bootstrap 95% CI
# =====================================================================================

fig,ax = plt.subplots(

    1,
    2,

    figsize=(
        11,
        4.3
    )

)


x = runtime[
    "N"
].to_numpy()


# -------------------------------------------------------------------------
# Panel A
# -------------------------------------------------------------------------

exact_med = runtime[
    "exact_sec"
].to_numpy()


exact_err = np.vstack(

    [

        exact_med
        -
        runtime[
            "exact_q1"
        ].to_numpy(),

        runtime[
            "exact_q3"
        ].to_numpy()
        -
        exact_med

    ]

)


neural_med = runtime[
    "neural_sec"
].to_numpy()


neural_err = np.vstack(

    [

        neural_med
        -
        runtime[
            "neural_q1"
        ].to_numpy(),

        runtime[
            "neural_q3"
        ].to_numpy()
        -
        neural_med

    ]

)


ax[0].errorbar(

    x,
    exact_med,

    yerr=exact_err,

    fmt="o-",

    color="#D55E00",

    lw=2,

    capsize=3,

    label="Exact"

)


ax[0].errorbar(

    x,
    neural_med,

    yerr=neural_err,

    fmt="s-",

    color="#0072B2",

    lw=2,

    capsize=3,

    label="Neural"

)


ax[0].set_yscale(
    "log"
)


ax[0].set_xlabel(
    "Population size $N$"
)


ax[0].set_ylabel(
    "Complete-query runtime (seconds)"
)


ax[0].set_title(
    "(A) Complete-query runtime"
)


ax[0].grid(
    alpha=.15
)


ax[0].legend(
    frameon=False
)


# -------------------------------------------------------------------------
# Panel B
# -------------------------------------------------------------------------

speed = runtime[
    "speedup"
].to_numpy()


speed_err = np.vstack(

    [

        speed
        -
        runtime[
            "speedup_CI_low"
        ].to_numpy(),

        runtime[
            "speedup_CI_high"
        ].to_numpy()
        -
        speed

    ]

)


ax[1].errorbar(

    x,
    speed,

    yerr=speed_err,

    fmt="o-",

    color="#009E73",

    lw=2,

    capsize=3

)


ax[1].set_yscale(
    "log"
)


ax[1].set_xlabel(
    "Population size $N$"
)


ax[1].set_ylabel(
    "Exact / neural runtime"
)


ax[1].set_title(
    "(B) Complete-query speedup"
)


ax[1].grid(
    alpha=.15
)


plt.tight_layout()


plt.savefig(

    OUT
    /
    "runtime_CPU_publication.pdf",

    bbox_inches="tight"

)


plt.savefig(

    OUT
    /
    "runtime_CPU_publication.png",

    dpi=300,

    bbox_inches="tight"

)


plt.show()


# =====================================================================================
# 36. FIGURE 2 — ACCURACY-DEPENDENT BREAK-EVEN
# =====================================================================================

if len(
    br
):

    fig,ax = plt.subplots(

        figsize=(
            7,
            4.5
        )

    )


    for eps,z in br.groupby(
        "epsilon"
    ):

        ax.plot(

            z["N"],

            z["B_star"],

            "o-",

            lw=2,

            label=rf"$\varepsilon={eps:g}$"

        )


    ax.set_yscale(
        "log"
    )


    ax.set_xlabel(
        "Population size $N$"
    )


    ax.set_ylabel(
        r"Break-even evaluations "
        r"$B_\varepsilon^\star(N)$"
    )


    ax.grid(
        alpha=.15
    )


    ax.legend(
        frameon=False
    )


    plt.tight_layout()


    plt.savefig(

        OUT
        /
        "break_even_CPU.pdf",

        bbox_inches="tight"

    )


    plt.savefig(

        OUT
        /
        "break_even_CPU.png",

        dpi=300,

        bbox_inches="tight"

    )


    plt.show()


# =====================================================================================
# 37. FINAL CONSISTENCY AUDIT
# =====================================================================================

print(
    "\n"
    +
    "="*105
)


print(
    "FINAL CONSISTENCY AUDIT"
)


print(
    "="*105
)


for _,z in acc.iterrows():

    R = int(
        z["R"]
    )


    Cteach = float(
        z["Cteach_sec"]
    )


    training = float(
        z["training_sec"]
    )


    offline = float(
        z["offline_sec"]
    )


    if (
        not np.isfinite(training)
        or
        training<=0
    ):

        raise RuntimeError(
            f"Invalid training cost for R={R}: "
            f"{training}"
        )


    if not np.isclose(
        offline,
        Cteach+training
    ):

        raise RuntimeError(
            f"Offline cost mismatch at R={R}"
        )


    print(
        f"R={R:5d} | "
        f"Cteach={Cteach:10.1f}s | "
        f"training={training:9.1f}s | "
        f"offline={offline:10.1f}s | "
        "OK"
    )


# =====================================================================================
# 38. FINAL REPORT
# =====================================================================================

print(
    "\n"
    +
    "="*105
)


print(
    "EXPERIMENT 5.3-C COMPLETE — RESUMABLE CPU VERSION"
)


print(
    "="*105
)


print(
    "Persistent Google Drive folder:"
)


print(
    ROOT
)


print()


print(
    "Logical CPU cores:",
    CPU
)


print(
    "Exact-LU workers:",
    N_EXACT
)


print(
    "PyTorch threads:",
    torch.get_num_threads()
)


print()


print(
    "R grid:",
    R_GRID
)


print(
    "Training exact configurations:",
    RMAX
)


print(
    "Validation configurations:",
    N_VAL
)


print(
    "Test configurations:",
    N_TEST
)


print()


print(
    "Benchmark configurations/N:",
    BENCH_PER_N
)


print(
    "Exact technical repetitions/config:",
    EXACT_REPEATS
)


print(
    "Neural blocks/config:",
    NEURAL_BLOCKS
)


print(
    "Neural queries/block:",
    NEURAL_REPEATS
)


print(
    "Bootstrap replicates:",
    BOOTSTRAP_B
)


print(
    "Bootstrap unit:",
    "independent epidemic configuration"
)


print()


print(
    "Exact target chunk size:",
    EXACT_CHUNK
)


print(
    "Neural checkpoint interval:",
    f"{CHECKPOINT_EVERY} epochs"
)


print()


print(
    "Accuracy/offline-cost table:"
)

print(
    OUT
    /
    "accuracy_cost_by_R_CPU.tex"
)


print(
    "Runtime table:"
)

print(
    OUT
    /
    "runtime_CPU.tex"
)


print(
    "Break-even table:"
)

print(
    OUT
    /
    "break_even_CPU.tex"
)


print(
    "Runtime figure:"
)

print(
    OUT
    /
    "runtime_CPU_publication.pdf"
)


print(
    "Break-even figure:"
)

print(
    OUT
    /
    "break_even_CPU.pdf"
)


print(
    "\nAFTER A GOOGLE COLAB DISCONNECT:"
)


print(
    "1. Reconnect."
)


print(
    "2. Run THIS SAME CELL."
)


print(
    "3. Completed exact-target chunks are skipped."
)


print(
    "4. Interrupted neural fits resume from "
    "the latest 5-epoch checkpoint."
)


print(
    "5. Completed R values are skipped."
)


print(
    "6. Completed runtime benchmark configurations are skipped."
)


print(
    "7. All permanent files remain in Google Drive."
)


print(
    "="*105
)

Mounted at /content/drive
PERMANENT GOOGLE DRIVE DIRECTORY
/content/drive/MyDrive/StatisticalLearning/Experiment_5_3C_CPU_publication_resumable_v5
Logical CPU cores: 2
Exact-LU workers: 2
PyTorch threads: 2
R grid: (500, 1000, 2000, 5000, 10000)
Exact training configurations: 10000
Validation configurations: 200
Test configurations: 300
Benchmark configurations/N: 5
Exact timings/configuration: 5
Neural timing blocks/configuration: 5
Neural queries/block: 100
Training checkpoint every: 5 epochs
GPU used: NO

Generating deterministic designs...

Generating/loading exact datasets...
TRAIN_R10000     0:   50 | 2 exact workers
  saved permanently | 95.9s
TRAIN_R10000    50:  100 | 2 exact workers
  saved permanently | 85.3s
TRAIN_R10000   100:  150 | 2 exact workers
